# Dispatch GNN — Learn Continuous Dispatch from MILP Reports

**Goal**: Train a GNN that takes **EBM-generated binary decisions** `[Z, T, 7]` and **HTE zone embeddings** `[Z, T, 128]` as inputs, and predicts the **optimal continuous dispatch** `[Z, T, 11]` — replacing the Decoder + LP Worker stages.

## Architecture: DispatchGNN
1. **Input projection**: concat(binaries, embeddings) `[Z, T, 135]` -> `[Z, T, H]`
2. **Temporal GRU**: bidirectional per-zone temporal encoding
3. **Zone GAT layers**: multi-head attention across zones (spatial GNN)
4. **Output MLP**: predicts 11 continuous dispatch channels per (zone, timestep)

## Dispatch channels (11)
| Idx | Variable | Description |
|-----|----------|-------------|
| 0 | thermal | Thermal generation (MW) |
| 1 | nuclear | Nuclear generation (MW) |
| 2 | solar | Solar dispatch (MW) |
| 3 | wind | Wind dispatch (MW) |
| 4 | battery_charge | Battery charging (MW) |
| 5 | battery_discharge | Battery discharging (MW) |
| 6 | pumped_charge | Pumped hydro charging (MW) |
| 7 | pumped_discharge | Pumped hydro discharging (MW) |
| 8 | demand_response | DR shed (MW) |
| 9 | unserved | Unserved energy (MW) |
| 10 | hydro_release | Hydro reservoir release (MW) |

## Data
- **5000 scenarios** from scenarios_v3
- **Reports**: `benchmark/outputs/scenarios_v3/reports/`
- **HTE embeddings**: `benchmark/outputs/encoders/hierarchical_temporal_v3/embeddings_v3/`
- **Trained EBM**: `benchmark/outputs/ebm_models/ebm_v3/ebm_v3_final.pt`

In [ ]:
# ============================================================
# Cell 1: Setup - Mount Drive, verify GPU
# ============================================================
from google.colab import drive
drive.mount('/content/drive')

import torch
print(f'PyTorch {torch.__version__}, CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'  GPU: {torch.cuda.get_device_name(0)}')
    print(f'  Memory: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB')

In [ ]:
# ============================================================
# Cell 2: Imports & Configuration
# ============================================================
import sys, os, json, time, glob
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

# Add benchmark to path
REPO = Path('/content/drive/MyDrive/benchmark')
sys.path.insert(0, str(REPO))

from src.gnn.dispatch_model import DispatchGNN, DISPATCH_CHANNELS, N_DISPATCH
from src.gnn.dispatch_dataset import (
    DispatchDataset, dispatch_collate_fn, build_dispatch_dataloaders,
    compute_channel_stats, extract_dispatch_from_report,
    extract_binaries_from_report,
)
from src.gnn.dispatch_trainer import TrainConfig, run_training

# ── Paths ──
REPORTS_DIR   = REPO / 'outputs' / 'scenarios_v3' / 'reports'
EMBED_DIR     = REPO / 'outputs' / 'encoders' / 'hierarchical_temporal_v3' / 'embeddings_v3'
EBM_PATH      = REPO / 'outputs' / 'ebm_models' / 'ebm_v3' / 'ebm_v3_final.pt'
OUTPUT_DIR    = REPO / 'outputs' / 'gnn_dispatch'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

# Verify paths
for p, label in [(REPORTS_DIR, 'Reports'), (EMBED_DIR, 'Embeddings'), (EBM_PATH, 'EBM')]:
    print(f'{label}: {"OK" if p.exists() else "MISSING"} -> {p}')

n_reports = len(list(REPORTS_DIR.glob('scenario_*.json'))) if REPORTS_DIR.exists() else 0
print(f'\nTotal reports: {n_reports}')

## Step A: Build DataLoaders
Load (binaries, embeddings, dispatch_targets) triplets from MILP reports + HTE embeddings.

In [ ]:
# ============================================================
# Cell 3: Build DataLoaders
# ============================================================
# Set max_scenarios=None to use all 5000, or a smaller number for quick testing
MAX_SCENARIOS = None  # set to e.g. 500 for a quick test run
BATCH_SIZE = 16
VAL_SPLIT = 0.1

train_loader, val_loader, dataset = build_dispatch_dataloaders(
    reports_dir=str(REPORTS_DIR),
    embeddings_dir=str(EMBED_DIR),
    n_timesteps=24,
    embed_dim=128,
    batch_size=BATCH_SIZE,
    val_split=VAL_SPLIT,
    num_workers=2,
    seed=42,
    max_scenarios=MAX_SCENARIOS,
)

# Inspect one batch
sample = next(iter(train_loader))
print(f'\nBatch shapes:')
print(f'  u_zt:      {sample["u_zt"].shape}')     # [B, Z, T, 7]
print(f'  h_zt:      {sample["h_zt"].shape}')     # [B, Z, T, 128]
print(f'  y_zt:      {sample["y_zt"].shape}')     # [B, Z, T, 11]
print(f'  zone_mask: {sample["zone_mask"].shape}') # [B, Z]
print(f'  n_zones:   {sample["n_zones"].tolist()}')

In [ ]:
# ============================================================
# Cell 4: Compute channel normalization stats
# ============================================================
print('Computing per-channel normalization stats from training data...')
channel_mean, channel_std = compute_channel_stats(train_loader)

print(f'\nChannel statistics (MW):')
print(f'{"Channel":>20s}  {"Mean":>10s}  {"Std":>10s}')
print('-' * 45)
for i, name in enumerate(DISPATCH_CHANNELS):
    print(f'{name:>20s}  {channel_mean[i]:10.1f}  {channel_std[i]:10.1f}')

## Step B: Train DispatchGNN

In [ ]:
# ============================================================
# Cell 5: Training configuration
# ============================================================
cfg = TrainConfig(
    # Model architecture
    n_binary_features=7,
    embed_dim=128,
    hidden_dim=256,
    n_dispatch=N_DISPATCH,
    gru_layers=2,
    n_gat_layers=3,
    n_heads=8,
    dropout=0.1,

    # Training
    epochs=60,
    lr=3e-4,
    weight_decay=1e-4,
    warmup_epochs=5,
    grad_clip=1.0,
    batch_size=BATCH_SIZE,

    # Channel weights: upweight expensive resources
    # thermal, nuclear, solar, wind, batt_ch, batt_dis,
    # pump_ch, pump_dis, dr, unserved, hydro
    channel_weights=[2.0, 2.0, 1.0, 1.0, 1.5, 1.5,
                     1.5, 1.5, 1.0, 3.0, 1.0],

    # Paths
    repo_path=str(REPO),
    output_dir='outputs/gnn_dispatch',

    device=DEVICE,
)

print('Training config:')
for k, v in vars(cfg).items():
    if not k.startswith('_'):
        print(f'  {k}: {v}')

In [ ]:
# ============================================================
# Cell 6: Run Training
# ============================================================
model, history = run_training(
    cfg,
    train_loader,
    val_loader,
    channel_mean=channel_mean,
    channel_std=channel_std,
)
print(f'\nTraining complete. Model saved to {OUTPUT_DIR}')

In [ ]:
# ============================================================
# Cell 7: Plot training curves
# ============================================================
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

epochs = [h['epoch'] for h in history]

# Loss curves
axes[0].plot(epochs, [h['train_loss'] for h in history], label='Train')
axes[0].plot(epochs, [h['val_loss'] for h in history], label='Val')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].set_title('Training & Validation Loss')
axes[0].legend()
axes[0].set_yscale('log')
axes[0].grid(True, alpha=0.3)

# Overall MAE
axes[1].plot(epochs, [h['mae_overall'] for h in history])
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('MAE (MW)')
axes[1].set_title('Overall MAE')
axes[1].grid(True, alpha=0.3)

# Learning rate
axes[2].plot(epochs, [h['lr'] for h in history])
axes[2].set_xlabel('Epoch')
axes[2].set_ylabel('LR')
axes[2].set_title('Learning Rate Schedule')
axes[2].set_yscale('log')
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(str(OUTPUT_DIR / 'training_curves.png'), dpi=150, bbox_inches='tight')
plt.show()

# Per-channel MAE at final epoch
final = history[-1]
channels = DISPATCH_CHANNELS
mae_vals = [final.get(f'mae_{c}', 0) for c in channels]

fig, ax = plt.subplots(figsize=(12, 5))
bars = ax.bar(channels, mae_vals, color='steelblue')
ax.set_ylabel('MAE (MW)')
ax.set_title('Per-Channel MAE at Final Epoch')
ax.set_xticklabels(channels, rotation=45, ha='right')
for bar, val in zip(bars, mae_vals):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
            f'{val:.0f}', ha='center', va='bottom', fontsize=9)
plt.tight_layout()
plt.savefig(str(OUTPUT_DIR / 'per_channel_mae.png'), dpi=150, bbox_inches='tight')
plt.show()

## Step C: Evaluation - Compare GNN Dispatch vs MILP Ground Truth

In [ ]:
# ============================================================
# Cell 8: Detailed evaluation on validation set
# ============================================================
from src.gnn.dispatch_trainer import validate
from src.gnn.dispatch_trainer import MaskedMSELoss

# Reload best model
best_ckpt = torch.load(OUTPUT_DIR / 'dispatch_gnn_best.pt', map_location=DEVICE, weights_only=False)
model.load_state_dict(best_ckpt['model_state_dict'])
model.eval()
print(f'Loaded best model from epoch {best_ckpt["epoch"]}, val_loss={best_ckpt["val_loss"]:.4f}')

# Detailed per-channel metrics
loss_fn = MaskedMSELoss(
    channel_weights=torch.ones(N_DISPATCH, device=DEVICE),
    normalize_mean=channel_mean.to(DEVICE),
    normalize_std=channel_std.to(DEVICE),
)

val_metrics = validate(model, val_loader, loss_fn, DEVICE)

print(f'\nValidation Metrics:')
print(f'  Overall MAE: {val_metrics["mae_overall"]:.1f} MW')
print(f'\n  Per-channel MAE (MW):')
for c in DISPATCH_CHANNELS:
    print(f'    {c:>20s}: {val_metrics.get(f"mae_{c}", 0):.1f}')

In [ ]:
# ============================================================
# Cell 9: Visualize dispatch predictions vs MILP ground truth
# ============================================================
model.eval()

# Get a validation batch
val_batch = next(iter(val_loader))
u_zt = val_batch['u_zt'].to(DEVICE)
h_zt = val_batch['h_zt'].to(DEVICE)
y_zt = val_batch['y_zt']
zone_mask = val_batch['zone_mask'].to(DEVICE)
scenario_ids = val_batch['scenario_ids']
n_zones = val_batch['n_zones']

with torch.no_grad():
    pred = model(u_zt, h_zt, zone_mask).cpu()

# Plot first scenario in batch
idx = 0
Z = n_zones[idx].item()
sc_id = scenario_ids[idx]
T = 24
t_axis = np.arange(T)

# Aggregate over zones (sum)
pred_agg = pred[idx, :Z, :, :].sum(dim=0).numpy()   # [T, 11]
true_agg = y_zt[idx, :Z, :, :].sum(dim=0).numpy()   # [T, 11]

# Plot key channels
key_channels = [0, 1, 2, 3, 4, 5, 10]  # thermal, nuclear, solar, wind, batt_ch, batt_dis, hydro
fig, axes = plt.subplots(len(key_channels), 1, figsize=(14, 3*len(key_channels)), sharex=True)

for i, c_idx in enumerate(key_channels):
    ax = axes[i]
    ax.plot(t_axis, true_agg[:, c_idx], 'b-', linewidth=2, label='MILP (true)')
    ax.plot(t_axis, pred_agg[:, c_idx], 'r--', linewidth=2, label='GNN (pred)')
    ax.set_ylabel('MW')
    ax.set_title(f'{DISPATCH_CHANNELS[c_idx]} - {sc_id} ({Z} zones)')
    ax.legend(loc='upper right')
    ax.grid(True, alpha=0.3)

axes[-1].set_xlabel('Time step (h)')
plt.tight_layout()
plt.savefig(str(OUTPUT_DIR / 'dispatch_comparison.png'), dpi=150, bbox_inches='tight')
plt.show()

## Step D: Pipeline Integration

Use `GNNDispatchPredictor` as a **drop-in replacement** for the Decoder + LP Worker.

**Original pipeline**: HTE -> EBM -> Decoder -> LP Worker -> dispatch

**New pipeline**: HTE -> EBM -> **GNN Dispatcher** -> dispatch

In [ ]:
# ============================================================
# Cell 10: Pipeline integration - GNN replaces Decoder + LP
# ============================================================
from src.gnn.dispatch_predictor import GNNDispatchPredictor
from src.ebm.model_v3 import TrajectoryZonalEBM
from src.ebm.sampler_v3 import NormalizedTemporalLangevinSampler

# Load GNN Dispatch Predictor
gnn_model_path = str(OUTPUT_DIR / 'dispatch_gnn_best.pt')
predictor = GNNDispatchPredictor(gnn_model_path, device=DEVICE)

# Load EBM for binary generation
ebm = TrajectoryZonalEBM(embed_dim=128, n_features=7).to(DEVICE)
ebm_ckpt = torch.load(str(EBM_PATH), map_location=DEVICE, weights_only=False)
if 'model_state_dict' in ebm_ckpt:
    ebm.load_state_dict(ebm_ckpt['model_state_dict'])
else:
    ebm.load_state_dict(ebm_ckpt)
ebm.eval()
print(f'EBM loaded: {sum(p.numel() for p in ebm.parameters()):,} params')

# Create Langevin sampler
sampler = NormalizedTemporalLangevinSampler(
    model=ebm,
    n_features=7,
    num_steps=100,
    step_size=0.05,
    noise_scale=0.50,
    temp_min=0.1,
    temp_max=1.0,
    init_mode='bernoulli',
    mode='infer',
    device=DEVICE,
)
print('Sampler ready')

In [ ]:
# ============================================================
# Cell 11: Full pipeline demo - EBM -> GNN Dispatch
# ============================================================
import time

# Pick a random validation scenario
demo_batch = next(iter(val_loader))
demo_idx = 0
demo_sc = demo_batch['scenario_ids'][demo_idx]
demo_h = demo_batch['h_zt'][demo_idx]       # [Z, T, D]
demo_y = demo_batch['y_zt'][demo_idx]       # [Z, T, 11]
demo_nz = demo_batch['n_zones'][demo_idx].item()
demo_mask = demo_batch['zone_mask'][demo_idx]  # [Z]

Z = demo_h.shape[0]
T = demo_h.shape[1]

print(f'Demo scenario: {demo_sc} ({demo_nz} zones, {T} timesteps)')

# Step 1: EBM sampling (generate binary candidates)
t0 = time.perf_counter()
h_zt_batch = demo_h.unsqueeze(0).to(DEVICE)       # [1, Z, T, D]
zm_batch = demo_mask.unsqueeze(0).to(DEVICE)       # [1, Z]

n_samples = 5
best_energy = float('inf')
best_u_bin = None

for s in range(n_samples):
    u_bin = sampler.sample_binary(
        h_zt=h_zt_batch,
        zone_mask=zm_batch,
        binarize='bernoulli',
        threshold=0.5,
    )  # [1, Z, T, 7]
    with torch.no_grad():
        energy = ebm(u_bin, h_zt_batch, zm_batch).item()
    if energy < best_energy:
        best_energy = energy
        best_u_bin = u_bin.squeeze(0).cpu()  # [Z, T, 7]

t_ebm = time.perf_counter() - t0
print(f'  EBM sampling: {t_ebm:.2f}s ({n_samples} samples, best energy={best_energy:.2f})')

# Step 2: GNN dispatch prediction (replaces decoder + LP)
t0 = time.perf_counter()
result = predictor.predict(
    u_bin=best_u_bin,
    h_zt=demo_h,
    zone_mask=demo_mask[:demo_nz],
    scenario_id=demo_sc,
)
t_gnn = time.perf_counter() - t0
print(f'  GNN dispatch: {t_gnn*1000:.1f}ms')
print(f'  Total pipeline: {t_ebm + t_gnn:.2f}s')

# Compare with MILP ground truth
gnn_dispatch = torch.zeros(Z, T, N_DISPATCH)
for c_idx, ch in enumerate(DISPATCH_CHANNELS):
    if ch in result.continuous_vars:
        arr = result.continuous_vars[ch]
        gnn_dispatch[:arr.shape[0], :arr.shape[1], c_idx] = torch.from_numpy(arr)

# Error analysis
valid_mask = demo_mask[:Z].unsqueeze(-1).unsqueeze(-1).float()  # [Z, 1, 1]
ae = (gnn_dispatch[:demo_nz] - demo_y[:demo_nz]).abs()
print(f'\n  Per-channel MAE vs MILP:')
for c_idx, ch in enumerate(DISPATCH_CHANNELS):
    mae = ae[:, :, c_idx].mean().item()
    print(f'    {ch:>20s}: {mae:.1f} MW')

In [ ]:
# ============================================================
# Cell 12: Speed comparison - GNN vs Decoder+LP pipeline
# ============================================================
import time

# Benchmark GNN dispatch on multiple scenarios
n_benchmark = min(50, len(val_loader.dataset))
gnn_times = []

for batch in val_loader:
    u_zt = batch['u_zt'].to(DEVICE)
    h_zt = batch['h_zt'].to(DEVICE)
    zone_mask = batch['zone_mask'].to(DEVICE)

    # Time the batch prediction
    torch.cuda.synchronize() if DEVICE == 'cuda' else None
    t0 = time.perf_counter()
    with torch.no_grad():
        pred = model(u_zt, h_zt, zone_mask)
    torch.cuda.synchronize() if DEVICE == 'cuda' else None
    elapsed = time.perf_counter() - t0

    per_scenario = elapsed / u_zt.shape[0]
    gnn_times.extend([per_scenario] * u_zt.shape[0])
    if len(gnn_times) >= n_benchmark:
        break

gnn_times = gnn_times[:n_benchmark]
mean_gnn = np.mean(gnn_times) * 1000  # ms
std_gnn = np.std(gnn_times) * 1000

print(f'GNN dispatch time per scenario: {mean_gnn:.1f} +/- {std_gnn:.1f} ms')
print(f'Typical LP Worker time: ~2-10 seconds per scenario')
print(f'Speedup: ~{2000/mean_gnn:.0f}x - {10000/mean_gnn:.0f}x')

In [ ]:
# ============================================================
# Cell 13: Scatter plot - GNN predictions vs MILP for each channel
# ============================================================
model.eval()

# Collect predictions over full validation set
all_pred = []
all_true = []
all_masks = []

for batch in val_loader:
    u_zt = batch['u_zt'].to(DEVICE)
    h_zt = batch['h_zt'].to(DEVICE)
    y_zt = batch['y_zt']
    zone_mask = batch['zone_mask'].to(DEVICE)

    with torch.no_grad():
        pred = model(u_zt, h_zt, zone_mask).cpu()

    all_pred.append(pred)
    all_true.append(y_zt)
    all_masks.append(batch['zone_mask'])

all_pred = torch.cat(all_pred, dim=0)
all_true = torch.cat(all_true, dim=0)
all_masks = torch.cat(all_masks, dim=0)

# Scatter plots for key channels
key_ch = [0, 1, 2, 3, 10]  # thermal, nuclear, solar, wind, hydro
fig, axes = plt.subplots(1, len(key_ch), figsize=(4*len(key_ch), 4))

for i, c_idx in enumerate(key_ch):
    ax = axes[i]
    mask_flat = all_masks.unsqueeze(-1).unsqueeze(-1).expand_as(all_pred[:,:,:,c_idx:c_idx+1]).squeeze(-1)
    p = all_pred[:, :, :, c_idx][mask_flat > 0].numpy().flatten()
    t = all_true[:, :, :, c_idx][mask_flat > 0].numpy().flatten()

    # Subsample for plotting
    n_pts = min(5000, len(p))
    idx = np.random.choice(len(p), n_pts, replace=False)

    ax.scatter(t[idx], p[idx], alpha=0.1, s=2)
    lim = max(t[idx].max(), p[idx].max()) * 1.05
    ax.plot([0, lim], [0, lim], 'r--', linewidth=1)
    ax.set_xlabel('MILP (MW)')
    ax.set_ylabel('GNN (MW)')
    ax.set_title(DISPATCH_CHANNELS[c_idx])
    ax.set_aspect('equal')
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(str(OUTPUT_DIR / 'scatter_comparison.png'), dpi=150, bbox_inches='tight')
plt.show()

## Step E: Full Pipeline Evaluation on Evaluation Families

Run the GNN-based pipeline on evaluation families (same scenarios used for Decoder+LP eval) to produce comparable results.

In [ ]:
# ============================================================
# Cell 14: Evaluate on a scenario family with GNN pipeline
# ============================================================
from tqdm.auto import tqdm

def evaluate_gnn_pipeline(
    predictor: GNNDispatchPredictor,
    sampler: NormalizedTemporalLangevinSampler,
    ebm: TrajectoryZonalEBM,
    reports_dir: Path,
    embeddings_dir: Path,
    max_scenarios: int = 100,
    n_ebm_samples: int = 5,
    device: str = 'cuda',
):
    """
    Evaluate the EBM -> GNN pipeline on scenarios.
    Uses MILP ground-truth binaries (teacher forcing) for fair comparison.
    """
    from src.gnn.dispatch_dataset import (
        extract_binaries_from_report, extract_dispatch_from_report,
    )

    report_files = sorted(reports_dir.glob('scenario_*.json'))[:max_scenarios]
    results = []

    for rpath in tqdm(report_files, desc='GNN Pipeline Eval'):
        sc_id = rpath.stem

        # Load report
        with open(rpath) as f:
            report = json.load(f)

        # Extract ground truth
        u_gt = extract_binaries_from_report(report, 24)
        dispatch_result = extract_dispatch_from_report(report, 24)
        if u_gt is None or dispatch_result is None:
            continue
        y_gt, zone_names = dispatch_result
        Z, T = u_gt.shape[0], u_gt.shape[1]

        # Load embedding
        emb_path = None
        for root, _, files in os.walk(str(embeddings_dir)):
            for f in files:
                if sc_id in f:
                    emb_path = os.path.join(root, f)
                    break
            if emb_path:
                break
        if emb_path is None:
            continue

        data = np.load(emb_path, allow_pickle=True)
        h_zt = torch.from_numpy(data['zones']).float()  # [Z_emb, T, D]

        # Align zone dims
        if h_zt.shape[0] < Z:
            h_zt = torch.nn.functional.pad(h_zt, (0,0,0,0,0,Z-h_zt.shape[0]))
        elif h_zt.shape[0] > Z:
            h_zt = h_zt[:Z]

        zone_mask = torch.ones(Z)

        # GNN prediction using GT binaries (teacher forcing)
        t0 = time.perf_counter()
        gnn_result = predictor.predict(
            u_bin=u_gt,
            h_zt=h_zt,
            zone_mask=zone_mask,
            zone_names=zone_names,
            scenario_id=sc_id,
        )
        t_gnn = time.perf_counter() - t0

        # Compute MAE vs MILP
        gnn_dispatch = torch.zeros(Z, T, N_DISPATCH)
        for c_idx, ch in enumerate(DISPATCH_CHANNELS):
            if ch in gnn_result.continuous_vars:
                arr = gnn_result.continuous_vars[ch]
                gnn_dispatch[:arr.shape[0], :arr.shape[1], c_idx] = torch.from_numpy(arr)

        mae = (gnn_dispatch - y_gt).abs().mean().item()
        milp_obj = report.get('mip', {}).get('objective', 0)

        results.append({
            'scenario_id': sc_id,
            'n_zones': Z,
            'mae_overall': mae,
            'gnn_time': t_gnn,
            'milp_objective': milp_obj,
            'milp_solve_time': report.get('mip', {}).get('solve_seconds', 0),
        })

    return results


# Run evaluation
eval_results = evaluate_gnn_pipeline(
    predictor, sampler, ebm,
    REPORTS_DIR, EMBED_DIR,
    max_scenarios=200,
    device=DEVICE,
)

# Summary stats
mae_vals = [r['mae_overall'] for r in eval_results]
gnn_times = [r['gnn_time'] for r in eval_results]
milp_times = [r['milp_solve_time'] for r in eval_results]

print(f'\n=== GNN Pipeline Evaluation ({len(eval_results)} scenarios) ===')
print(f'MAE overall: {np.mean(mae_vals):.1f} +/- {np.std(mae_vals):.1f} MW')
print(f'GNN time:    {np.mean(gnn_times)*1000:.1f} ms/scenario')
print(f'MILP time:   {np.mean(milp_times):.1f} s/scenario')
print(f'Speedup:     {np.mean(milp_times)/np.mean(gnn_times):.0f}x')

# Save results
with open(OUTPUT_DIR / 'gnn_eval_results.json', 'w') as f:
    json.dump(eval_results, f, indent=2)
print(f'\nResults saved to {OUTPUT_DIR / "gnn_eval_results.json"}')

In [ ]:
# ============================================================
# Cell 15: Summary
# ============================================================
print('=' * 60)
print('DISPATCH GNN - Training & Evaluation Complete')
print('=' * 60)
print(f'\nModel saved to:')
print(f'  Best:  {OUTPUT_DIR / "dispatch_gnn_best.pt"}')
print(f'  Final: {OUTPUT_DIR / "dispatch_gnn_final.pt"}')
print(f'\nTo use in pipeline:')
print(f'  from src.gnn.dispatch_predictor import GNNDispatchPredictor')
print(f'  predictor = GNNDispatchPredictor("{OUTPUT_DIR / "dispatch_gnn_best.pt"}")')
print(f'  result = predictor.predict(u_bin, h_zt, zone_mask)')
print(f'\nThis replaces: Decoder + LP Worker (stages 5-6 of pipeline)')